<a href="https://colab.research.google.com/github/nmwiley808/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project/blob/main/notebooks/03_CNN_GTZAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 – CNN Baseline on GTZAN (Timed Training)

## Description

This notebook trains an improved Convolutional Neural Network (CNN)
on GTZAN log-mel spectrogram features.

Pipeline:
1. Detect project path
2. Build feature cache (if missing)
3. Normalize input features
4. Train CNN with mini-batching
5. Track validation accuracy
6. Measure training runtime

Training time is recorded per epoch and overall.

In [2]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import librosa
import numpy as np
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import TensorDataset, DataLoader
import time

#  Hard-set correct project path
PROJECT_PATH = "/content/drive/MyDrive/csci198/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project"

if not os.path.exists(PROJECT_PATH):
    raise FileNotFoundError(f"Project path does not exist: {PROJECT_PATH}")

os.chdir(PROJECT_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Working directory:", os.getcwd())
print("Device:", device)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/csci198/csci198-Music-Intelligence-with-Deep-Learning-Senior-Project
Device: cuda


In [3]:
# Parameters
TARGET_SR = 22050
DURATION = 30
SAMPLES_PER_TRACK = TARGET_SR * DURATION
N_MELS = 128
BATCH_SIZE = 32
EPOCHS = 30

In [4]:
# Cache Paths
os.makedirs("data/processed", exist_ok=True)

X_PATH = "data/processed/X_gtzan.npy"
Y_PATH = "data/processed/Y_gtzan.npy"

In [6]:
# Build Cache If Missing
if not os.path.exists(X_PATH):

    print("Building GTZAN feature cache...")

    def load_audio(file_path):
        try:
            y, sr = librosa.load(file_path, sr=TARGET_SR)
            if len(y) > SAMPLES_PER_TRACK:
                y = y[:SAMPLES_PER_TRACK]
            else:
                y = np.pad(y, (0, SAMPLES_PER_TRACK - len(y)))
            return y
        except Exception as e:
            print(f"Warning: Could not load {file_path}. Skipping. Error: {e}")
            return None

    def extract_log_mel(y):
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=TARGET_SR,
            n_mels=N_MELS
        )
        return librosa.power_to_db(mel, ref=np.max)

    gtzan_root = "data/raw/gtzan"

    X = []
    labels = []

    for root, dirs, files in os.walk(gtzan_root):
        for file in tqdm(files):
            if file.endswith(".wav"):
                label = os.path.basename(root)
                file_path = os.path.join(root, file)

                y_audio = load_audio(file_path)
                if y_audio is not None:  # Only process if audio loaded successfully
                    log_mel = extract_log_mel(y_audio)

                    X.append(log_mel)
                    labels.append(label)

    X = np.array(X)
    le = LabelEncoder()
    y = le.fit_transform(labels)

    np.save(X_PATH, X)
    np.save(Y_PATH, y)

    print("Cache saved.")
else:
    print("Cache already exists.")

Building GTZAN feature cache...


100%|██████████| 2/2 [00:00<00:00, 29127.11it/s]
0it [00:00, ?it/s]
 49%|████▉     | 49/100 [00:02<00:02, 20.90it/s]/tmp/ipykernel_296/1724101033.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=TARGET_SR)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


100%|██████████| 100/100 [00:11<00:00,  8.77it/s]
0it [00:00, ?it/s]
100%|██████████| 100/100 [00:00<00:00, 984578.40it/s]


Cache saved.


In [ ]:
# Load & Normalize Data
X = np.load(X_PATH)
Y = np.load(Y_PATH)

mean = np.mean(X)
std = np.std(X)
X = (X - mean) / (std + 1e-8)

print("Data normalized.")
print("X shape:", X.shape)

In [ ]:
# Train/Test Split + DataLoader
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = torch.tensor(X_train).unsqueeze(1).float()
X_test = torch.tensor(X_test).unsqueeze(1).float()
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE)

In [ ]:
# CNN Architecture
class CNN(nn.Module):
    def __init__(self, num_classes):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x

In [ ]:
# Training Loop w/ Timer
model = CNN(num_classes=len(np.unique(y))).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

best_accuracy = 0

training_start_time = time.time()

for epoch in range(EPOCHS):

    epoch_start = time.time()

    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    # Validation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    if accuracy > best_accuracy:
        best_accuracy = accuracy

    epoch_time = time.time() - epoch_start

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Val Acc: {accuracy:.4f} | "
          f"Epoch Time: {epoch_time:.2f}s")

total_training_time = time.time() - training_start_time

print("\nBest Validation Accuracy:", best_accuracy)
print(f"Total Training Time: {total_training_time:.2f} seconds")